In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
project_path = '/content/drive/MyDrive/Apple_Dataset'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Đã truy cập: {os.getcwd()}")
else:
    print("Không tìm thấy thư mục. Kiểm tra lại tên folder trên Drive nhé!")

Đã truy cập: /content/drive/MyDrive/Apple_Dataset


In [ ]:


file_name = 'detection.zip'
if os.path.exists(file_name):
    print(f"Tìm thấy file: {file_name}")
    # Liệt kê thông tin file
    stats = os.stat(file_name)
    print(f"Kích thước: {stats.st_size} bytes")
else:
    print(f"Không tìm thấy file '{file_name}' trong thư mục hiện tại.")
    print("Danh sách các file đang có:")
    print(os.listdir('.'))

Tìm thấy file: detection.zip
Kích thước: 619203388 bytes


In [ ]:
import os
import subprocess

print("Đang quét toàn bộ hệ thống để tìm 'detection.zip'...")

# Dùng lệnh find của Linux để rà soát toàn bộ thư mục /content (bao gồm cả Drive đã kết nối)
result = subprocess.run(['find', '/content', '-type', 'f', '-name', 'detection.zip'], capture_output=True, text=True)
paths = result.stdout.strip().split('\n')

if paths and paths[0]:
    zip_path = paths[0]
    print(f"Bingo! Đã tìm thấy file tại: {zip_path}")

    # Tạo thư mục đích và giải nén bằng đường dẫn tuyệt đối vừa tìm được
    extract_dir = '/content/drive/MyDrive/Apple_Dataset/dataset'
    os.makedirs(extract_dir, exist_ok=True)

    print(f"Đang tiến hành giải nén vào {extract_dir}...")
    !unzip -q "{zip_path}" -d "{extract_dir}"

    print("Giải nén thành công! Dữ liệu của bạn đã sẵn sàng.")
else:
    print("Hoàn toàn không tìm thấy 'detection.zip' ở bất kỳ đâu. Bạn hãy kiểm tra lại xem file đã tải lên Drive thành công 100% chưa nhé!")

Đang quét toàn bộ hệ thống để tìm 'detection.zip'...
Bingo! Đã tìm thấy file tại: /content/drive/MyDrive/Apple_Dataset/detection.zip
Đang tiến hành giải nén vào /content/drive/MyDrive/Apple_Dataset/apple_data...
Giải nén thành công! Dữ liệu của bạn đã sẵn sàng.


In [ ]:
# 1. Tạo folder mới của riêng ông trên Drive
!mkdir -p /content/drive/MyDrive/Apple_Dataset/data_v3

# 2. Copy toàn bộ quân lương từ folder của bạn sang folder mới
# Lưu ý: Thay đường dẫn '/content/drive/MyDrive/Final_Project/data' bằng đường dẫn chuẩn của cái Shortcut đó
print("--- ĐANG ĐIỀU QUÂN (COPY DATASET) ---")
!cp -r /content/drive/MyDrive/Apple_Dataset/data/* /content/drive/MyDrive/Apple_Dataset/data_v3/
print("✅ Đã chiếm hữu thành công! Giờ folder data_v3 là của ông.")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
--- ĐANG ĐIỀU QUÂN (COPY DATASET) ---
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
✅ Đã chiếm hữu thành công! Giờ folder data_v3 là của ông.


In [ ]:
import json
import os
import shutil

# Cấu hình đường dẫn hệ thống
BASE_PATH = '/content/drive/MyDrive/Apple_Dataset/data_v3'
LOCAL_TEMP_DIR = '/content/local_labels'

def convert_coco_to_yolo_academic(json_file, folder_name):
    """
    Chuyển đổi dữ liệu từ định dạng COCO sang YOLO cho bài toán nhận diện đa lớp.

    Args:
        json_file (str): Tên tệp tin chú thích định dạng JSON (COCO).
        folder_name (str): Tên thư mục tương ứng (train/valid/test).
    """
    json_path = os.path.join(BASE_PATH, json_file)
    local_output_dir = os.path.join(LOCAL_TEMP_DIR, folder_name)
    os.makedirs(local_output_dir, exist_ok=True)

    if not os.path.exists(json_path):
        print(f"Cảnh báo: Không tìm thấy tệp tin {json_file}")
        return

    with open(json_path, 'r') as f:
        data = json.load(f)

    # Thiết lập bản đồ phân loại (Category Mapping)
    # Loại bỏ các lớp không liên quan, tập trung vào Green_Apple (ID: 1) và Red_Apple (ID: 2)
    id_mapping = {1: 0, 2: 1} # Ánh xạ ID COCO sang Index YOLO (0 và 1)

    # Khởi tạo từ điển tra cứu hình ảnh
    image_metadata = {img['id']: img for img in data['images']}
    annotation_count = 0

    for ann in data['annotations']:
        category_id = ann['category_id']
        if category_id not in id_mapping:
            continue # Loại bỏ các đối tượng không thuộc phạm vi nghiên cứu

        img_info = image_metadata[ann['image_id']]
        width, height = img_info['width'], img_info['height']

        # Chuyển đổi tọa độ: [x_min, y_min, width, height] -> [x_center, y_center, w, h] (Chuẩn hóa 0-1)
        bbox = ann['bbox']
        x_center = (bbox[0] + bbox[2] / 2) / width
        y_center = (bbox[1] + bbox[3] / 2) / height
        norm_w = bbox[2] / width
        norm_h = bbox[3] / height

        # Ghi tệp tin nhãn tương ứng với tên hình ảnh
        file_name = os.path.splitext(img_info['file_name'])[0] + ".txt"
        label_file_path = os.path.join(local_output_dir, file_name)

        with open(label_file_path, 'a') as f:
            f.write(f"{id_mapping[category_id]} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")
        annotation_count += 1

    # Đồng bộ dữ liệu từ môi trường tạm thời lên Google Drive
    drive_output_path = os.path.join(BASE_PATH, folder_name, 'labels')
    if os.path.exists(drive_output_path):
        shutil.rmtree(drive_output_path)
    shutil.copytree(local_output_dir, drive_output_path)

    print(f"Hoàn tất: Đã xử lý {annotation_count} đối tượng cho tập dữ liệu {folder_name}.")

# Thực thi tiến trình xử lý dữ liệu
print("Bắt đầu quy trình tiền xử lý dữ liệu...")
convert_coco_to_yolo_academic('train_coco.json', 'train')
convert_coco_to_yolo_academic('valid_coco.json', 'valid')

Bắt đầu quy trình tiền xử lý dữ liệu...
Hoàn tất: Đã xử lý 48351 đối tượng cho tập dữ liệu train.
Hoàn tất: Đã xử lý 6802 đối tượng cho tập dữ liệu valid.


In [ ]:
!ls /content/drive/MyDrive/Apple_Dataset/ultralytics

CITATION.cff	 examples	 README.md	  ultralytics.egg-info
CONTRIBUTING.md  LICENSE	 README.zh-CN.md  yolo26n.pt
docker		 mkdocs.yml	 tests
docs		 pyproject.toml  ultralytics


In [ ]:
%cd /content/drive/MyDrive/Apple_Dataset/ultralytics
!pip install -e .

/content/drive/MyDrive/Apple_Dataset/ultralytics
Obtaining file:///content/drive/MyDrive/Apple_Dataset/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.41-0.editable-py3-none-any.whl size=23305 sha256=fdde206d1a59f88d1683fe3110f9a7f9af703feeab346a4f3a7774e1caa94b47
  Stored in directory: /tmp/pip-ephem-wheel-cache-ra4me2he/wheels/f3/38/40/34061b2290a84783ebeb24f157aed4740fc0095421d61e4e00
Successfully built ultralytics


In [ ]:
import os
import shutil
import random

print("--- BƯỚC 3: TỰ ĐỘNG CHIA TẬP TRAIN VÀ VALIDATION (80/20) ---")

# Đường dẫn gốc trên Drive của ông
dataset_dir = '/content/drive/MyDrive/Apple_Dataset/dataset'
images_src = os.path.join(dataset_dir, 'images')
labels_src = os.path.join(dataset_dir, 'labels')

# Tạo cấu trúc thư mục chuẩn YOLO
folders = [
    os.path.join(images_src, 'train'), os.path.join(images_src, 'val'),
    os.path.join(labels_src, 'train'), os.path.join(labels_src, 'val')
]
for f in folders: os.makedirs(f, exist_ok=True)

# Lấy danh sách file ảnh (chỉ lấy file, không lấy thư mục đã tạo)
all_imgs = [f for f in os.listdir(images_src) if f.endswith(('.jpg', '.png'))]
random.seed(42)
random.shuffle(all_imgs)

# Tính toán vị trí chia
split_idx = int(len(all_imgs) * 0.8)
train_list = all_imgs[:split_idx]
val_list = all_imgs[split_idx:]

def process_split(file_list, split_name):
    for img_name in file_list:
        # Đường dẫn nhãn tương ứng (đổi đuôi sang .txt)
        lbl_name = img_name.rsplit('.', 1)[0] + '.txt'

        # Di chuyển Ảnh
        if os.path.exists(os.path.join(images_src, img_name)):
            shutil.move(os.path.join(images_src, img_name), os.path.join(images_src, split_name, img_name))

        # Di chuyển Nhãn
        if os.path.exists(os.path.join(labels_src, lbl_name)):
            shutil.move(os.path.join(labels_src, lbl_name), os.path.join(labels_src, split_name, lbl_name))

print(f"Đang xử lý {len(train_list)} ảnh Train...")
process_split(train_list, 'train')
print(f"Đang xử lý {len(val_list)} ảnh Val...")
process_split(val_list, 'val')

print("🎉 BINGO! Dataset đã được chia và sắp xếp chuẩn chỉnh.")

--- BƯỚC 3: TỰ ĐỘNG CHIA TẬP TRAIN VÀ VALIDATION (80/20) ---
Đang xử lý 0 ảnh Train...
Đang xử lý 0 ảnh Val...
🎉 BINGO! Dataset đã được chia và sắp xếp chuẩn chỉnh.


In [ ]:
import ultralytics;
print(ultralytics.__file__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/content/drive/MyDrive/Apple_Dataset/ultralytics/ultralytics/__init__.py


In [ ]:
import inspect
from ultralytics.nn.modules.block import SCSA
sig = inspect.signature(SCSA.__init__)
print(f"Cấu trúc SCSA trong RAM hiện tại: {sig}")

Cấu trúc SCSA trong RAM hiện tại: (self, c1, c2, n=1)


In [ ]:
import torch
from ultralytics import RTDETR

# Cấu hình đường dẫn
MODEL_CONFIG = '/content/drive/MyDrive/Apple_Dataset/scripts/msrrt-detr.yaml'
DATA_CONFIG = '/content/drive/MyDrive/Apple_Dataset/data.yaml' # Thay bằng file thật của ông
PROJECT_DIR = '/content/drive/MyDrive/Apple_Dataset/output'

def main():
    # Xóa cache GPU trước khi chạy
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 1. Khởi tạo "Thể xác" MSRRT-DETR từ file thiết kế
    model = RTDETR(MODEL_CONFIG)

    # 2. Bơm "Linh hồn" (Tri thức nền tảng từ COCO, loại bỏ hoàn toàn rác của v3_final)
    # Hệ thống sẽ tự động tải file này về nếu chưa có
    model.load('rtdetr-l.pt')

    print("--- BẮT ĐẦU CHINH PHẠT MSRRT-DETR (KIẾN TRÚC CHUẨN) ---")

    # 3. Kích hoạt quá trình huấn luyện
    results = model.train(
        data=DATA_CONFIG,
        epochs=100,
        batch=16,                 # Colab Pro A100 cân tốt batch 16
        device=0,
        project=PROJECT_DIR,
        name='msrrt_detr_v4_real', # Đổi tên folder để không đụng chạm rác cũ

        # --- CHIẾN LƯỢC TĂNG CƯỜNG DỮ LIỆU (ONLINE AUGMENTATION) ---
        # A. Điều chỉnh không gian màu HSV
        hsv_h=0.015,              # Sắc độ (Hue)
        hsv_s=0.7,                # Độ bão hòa (Saturation)
        hsv_v=0.4,                # Độ sáng (Value)

        # B. Biến đổi hình học
        degrees=10.0,             # Xoay ngẫu nhiên
        translate=0.1,            # Tịnh tiến
        scale=0.5,                # Thu phóng
        fliplr=0.5,               # Lật ngang (50% tỷ lệ)
        flipud=0.0,               # Lật dọc

        # C. Mô phỏng che khuất (Occlusion & Complex Background)
        mosaic=1.0,               # Ép 100% dùng Mosaic
        mixup=0.1,                # 10% Mixup
        erasing=0.3,              # 30% Random Erasing

        # D. Tinh chỉnh giai đoạn cuối
        close_mosaic=15           # Đóng Mosaic ở 15 epoch cuối
    )

if __name__ == '__main__':
    main()

WARNING ⚠️ no model scale passed. Assuming scale='l'.
Transferred 26/662 items from pretrained weights
--- BẮT ĐẦU CHINH PHẠT MSRRT-DETR (KIẾN TRÚC CHUẨN) ---
New https://pypi.org/project/ultralytics/8.4.43 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Apple_Dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      78.4G      2.362     0.1331     0.8887         93        640: 100% ━━━━━━━━━━━━ 44/44 1.2it/s 36.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.7s/it 7.0s
                   all        100       6800    0.00308     0.0141   0.000707   0.000201

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      69.3G      2.909    0.05588      1.002       1558        640: 0% ──────────── 0/44  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100        73G       2.26     0.1599     0.6766        318        640: 100% ━━━━━━━━━━━━ 44/44 2.7it/s 16.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.6it/s 0.9s
                   all        100       6800    0.00332      0.013    0.00133   0.000268

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      69.2G      2.105     0.1991     0.7519       1549        640: 0% ──────────── 0/44  0.4s

...

     98/100      68.5G     0.4439     0.4365    0.03975        188        640: 100% ━━━━━━━━━━━━ 44/44 3.4it/s 12.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.5it/s 0.9s
                   all        100       6800      0.882      0.823      0.922      0.542

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     99/100      68.4G     0.4481       0.43    0.04515        864        640: 0% ──────────── 0/44  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      68.5G     0.4292     0.4348    0.03808         91        640: 100% ━━━━━━━━━━━━ 44/44 3.4it/s 12.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.4it/s 0.9s
                   all        100       6800      0.851      0.851      0.922       0.56

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    100/100      67.9G     0.4278     0.4446    0.03818        847        640: 0% ──────────── 0/44  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100      68.6G     0.4359     0.4334    0.03899        134        640: 100% ━━━━━━━━━━━━ 44/44 3.4it/s 12.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.4it/s 0.9s
                   all        100       6800      0.855      0.836      0.914      0.564

100 epochs completed in 0.516 hours.
Optimizer stripped from /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v4_real-5/weights/last.pt, 72.5MB
Optimizer stripped from /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v4_real-5/weights/best.pt, 72.5MB

Validating /content/drive/MyDrive/Apple_Dataset/output/msrrt_detr_v4_real-5/weights/best.pt...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
msrrt-detr summary: 195 layers, 35,813,346 parameters, 0 gradients, 100.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.5s/it 10.0s
      